<a href="https://colab.research.google.com/github/RajMK420/Healthcare_Data_Cleaning.ipynb/blob/main/Pandas_Assignment_DataMerging_%5BMunna_kumar%5D.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 1: Data Preparation and Missing Value Handling
In this stage, we focus on data integrity by cleaning the students DataFrame before merging.

In [4]:
import pandas as pd
import numpy as np

# 1. Create DataFrames
students_data = {
    'student_id': [101, 102, 103, 104, 105, 106, 107],
    'name': ['Alice', 'Bob', None, 'David', 'Emma', 'Frank', 'Grace'],
    'email': ['alice@email.com', 'bob@email.com', 'charlie@email.com', None, 'emma@email.com', 'frank@email.com', 'grace@email.com'],
    'city': ['Mumbai', 'Delhi', 'Bangalore', 'Mumbai', None, 'Chennai', 'Delhi']
}

enrollments_data = {
    'student_id': [101, 102, 103, 105, 108, 109],
    'course_name': ['Python', 'Data Science', 'Python', 'Machine Learning', 'AI', 'Python'],
    'enrollment_date': ['2024-01-15', '2024-01-20', '2024-02-01', '2024-02-10', '2024-02-15', '2024-03-01']
}

scores_data = {
    'student_id': [101, 102, 104, 105, 106],
    'exam_score': [85, 92, 78, 88, 95]
}

df_students = pd.DataFrame(students_data)
df_enroll = pd.DataFrame(enrollments_data)
df_scores = pd.DataFrame(scores_data)

# 2. Null Value Analysis
print("=== TASK 1: Null Value Analysis ===")
null_counts = df_students.isnull().sum()
null_pct = (df_students.isnull().sum() / len(df_students)) * 100
for col in df_students.columns:
    print(f"Column: {col}, Nulls: {null_counts[col]} ({null_pct[col]:.2f}%)")

# 3. Data Cleaning
df_students['city'] = df_students['city'].fillna('Unknown')
df_students = df_students.dropna(subset=['name'])

print("\nCleaned Students DataFrame:")
print(df_students)

=== TASK 1: Null Value Analysis ===
Column: student_id, Nulls: 0 (0.00%)
Column: name, Nulls: 1 (14.29%)
Column: email, Nulls: 1 (14.29%)
Column: city, Nulls: 1 (14.29%)

Cleaned Students DataFrame:
   student_id   name            email     city
0         101  Alice  alice@email.com   Mumbai
1         102    Bob    bob@email.com    Delhi
3         104  David             None   Mumbai
4         105   Emma   emma@email.com  Unknown
5         106  Frank  frank@email.com  Chennai
6         107  Grace  grace@email.com    Delhi


Task 2: Multiple Join Operations
This section explores how different joins affect the resulting dataset. Note that after Task 1, Student 103 (Charlie) was dropped because their name was missing.

In [5]:
print("\n=== TASK 2: Join Operations ===")

# --- Inner Join ---
inner_merge = pd.merge(df_students, df_enroll, on='student_id', how='inner')
print(f"Inner Join: {len(inner_merge)} rows.")
# Logic: 104, 106, 107 are excluded because they don't exist in the enrollments table.

# --- Left Join ---
left_merge = pd.merge(df_students, df_enroll, on='student_id', how='left')
print(f"Left Join: {len(left_merge)} rows.")
# Logic: 104, 106, 107 have nulls because they are in Students but didn't enroll in a course.

# --- Right Join ---
right_merge = pd.merge(df_students, df_enroll, on='student_id', how='right')
print(f"Right Join: {len(right_merge)} rows.")
# IDs 108 and 109 appear without names because they are in Enrollments but not in the Students table.

# --- Full Outer Join ---
outer_merge = pd.merge(df_students, df_enroll, on='student_id', how='outer', indicator=True)
print(f"Outer Join: {len(outer_merge)} rows.")

# Rows where name OR course is null
null_rows = outer_merge[outer_merge['name'].isna() | outer_merge['course_name'].isna()]
print("\nRows with missing info (Name or Course):")
print(null_rows[['student_id', 'name', 'course_name']])

print("\nMerge Source Distribution:")
print(outer_merge['_merge'].value_counts())


=== TASK 2: Join Operations ===
Inner Join: 3 rows.
Left Join: 6 rows.
Right Join: 6 rows.
Outer Join: 9 rows.

Rows with missing info (Name or Course):
   student_id   name course_name
2         103    NaN      Python
3         104  David         NaN
5         106  Frank         NaN
6         107  Grace         NaN
7         108    NaN          AI
8         109    NaN      Python

Merge Source Distribution:
_merge
left_only     3
right_only    3
both          3
Name: count, dtype: int64


Task 3: Lookup Operation and Automation
Here we compare the .map() method (fast for 1-to-1 lookups) against the more flexible .merge() method.

In [6]:
print("\n=== TASK 3: Lookup and Automation ===")

# 1. Lookup with .map()
score_map = df_scores.set_index('student_id')['exam_score'].to_dict()
df_students['exam_score'] = df_students['student_id'].map(score_map)
print("Students with Scores (via map):")
print(df_students[['student_id', 'name', 'exam_score']])

# 2. Automation Function
def auto_merge(df1, df2, join_type, key_column):
    merged = pd.merge(df1, df2, on=key_column, how=join_type)
    return {
        'result_df': merged,
        'row_count': len(merged),
        'join_type': join_type
    }

# Testing the function
test_inner = auto_merge(df_students, df_enroll, 'inner', 'student_id')
test_left = auto_merge(df_students, df_enroll, 'left', 'student_id')

print(f"\nAutomation Test (Inner): {test_inner['row_count']} rows")
print(f"Automation Test (Left): {test_left['row_count']} rows")


=== TASK 3: Lookup and Automation ===
Students with Scores (via map):
   student_id   name  exam_score
0         101  Alice        85.0
1         102    Bob        92.0
3         104  David        78.0
4         105   Emma        88.0
5         106  Frank        95.0
6         107  Grace         NaN

Automation Test (Inner): 3 rows
Automation Test (Left): 6 rows


Student Name: [MUNNA KUMAR]
Notebook Link: [Google Drive/GitHub URL]
All Tasks Completed: Yes
Additional Notes: [ask 1: Successfully cleaned the Students DataFrame by dropping 1 null name (Charlie) and filling missing cities with 'Unknown'.

Task 2: Demonstrated Inner, Left, Right, and Outer joins. Observed that Right joins introduced new student_ids (108, 109) that were not in the primary student list.

Task 3: Successfully implemented a lookup using .map() for exam scores and created a reusable auto_merge function for future automation.]
